# Proyecto 8 — Clasificación automática de SMS no deseados
**Profundización III · Inteligencia Artificial · Tercer corte: integración, profundización y cierre responsable**

Este notebook continúa exactamente el flujo del Corte 2 (misma lectura, misma depuración de duplicados y **misma partición estratificada 70/15/15 con `random_state=42`**) y agrega lo que pide la guía para el Corte 3:

| Requisito de la guía (Corte 3) | Sección |
|---|---|
| Profundización pertinente | 5 (representaciones + modelo moderno acotado), 6 (calibración), 7 (umbral por costo) |
| Evaluación final en test reservado | 8 |
| Comparación contra línea base (Corte 1) y modelo provisional (Corte 2) | 8 |
| Qué cambia al ajustar el umbral | 9 |
| Términos que dominan y posible sesgo temporal | 10 |
| Pruebas de robustez / sensibilidad | 11 |
| Función de inferencia para mensajes nuevos + demo | 12 |
| Model card y ficha del dataset | 13 |
| Paquete de resultados para el informe y la presentación | 14 |

**Regla de oro:** todas las decisiones (representación, calibración, umbral) se toman con entrenamiento + validación. El conjunto de **test** se usa una sola vez, en la sección 8, con todo ya fijado.

**Cómo correrlo:** en Colab, `Entorno de ejecución → Ejecutar todas`. Tarda pocos minutos (la comparación con embeddings es opcional y se salta sola si no se puede cargar el modelo). Al final se descarga `resultados_corte3.zip`.

## 1. Librerías y configuración

In [ ]:
from pathlib import Path
import io, os, re, json, zipfile, urllib.request, textwrap, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.dummy import DummyClassifier
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (precision_score, recall_score, f1_score, accuracy_score,
                             confusion_matrix, average_precision_score,
                             precision_recall_curve, brier_score_loss, classification_report)
import joblib, sklearn

warnings.filterwarnings("ignore", category=FutureWarning)
try:
    display
except NameError:            # permite correrlo fuera de Jupyter
    def display(x): print(x)

SEED = 42
BASE = Path("/content") if Path("/content").exists() else Path(".")
DATA_DIR = BASE / "data"
REPORTS  = BASE / "reports"
FIGS     = REPORTS / "figuras"
MODELS   = BASE / "models"
SRC      = BASE / "src"
for d in (DATA_DIR, REPORTS, FIGS, MODELS, SRC):
    d.mkdir(parents=True, exist_ok=True)

# Paleta sobria y consistente para todas las figuras
C_HAM, C_SPAM, C_GRIS, C_OSC = "#6E9C80", "#E0533D", "#9AA0A6", "#2B2F33"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "font.size": 10})

def guardar_fig(nombre):
    plt.tight_layout()
    plt.savefig(FIGS / nombre, dpi=170, bbox_inches="tight")
    plt.show()

RESUMEN = {"sklearn": sklearn.__version__}   # aquí se acumulan los números para la presentación
print("scikit-learn", sklearn.__version__, "| carpeta base:", BASE.resolve())

## 2. Datos: misma lectura y depuración del Corte 2
Se usa la misma lectura del Corte 2 (pandas, separador tabulador) para que la partición sea idéntica. Esa lectura entrega 5.572 filas (UCI documenta 5.574: pandas une dos líneas que contienen comillas). Luego se eliminan los textos duplicados exactos conservando la primera aparición → **5.169 mensajes únicos**.

In [ ]:
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"
URL_ALT = "https://archive.ics.uci.edu/static/public/228/sms+spam+collection.zip"
ruta = DATA_DIR / "SMSSpamCollection"

if not ruta.exists():
    for url in (URL, URL_ALT):
        try:
            print("Descargando desde", url)
            with urllib.request.urlopen(url, timeout=60) as r:
                zbytes = r.read()
            with zipfile.ZipFile(io.BytesIO(zbytes)) as zf:
                miembro = next(n for n in zf.namelist() if n.endswith("SMSSpamCollection"))
                ruta.write_bytes(zf.read(miembro))
            break
        except Exception as e:
            print("  falló:", e)
assert ruta.exists(), "No se pudo descargar el dataset. Súbelo a data/SMSSpamCollection."

df = pd.read_csv(ruta, sep="\t", header=None, names=["etiqueta", "mensaje"], encoding="utf-8")
print("Filas leídas:", len(df))
datos = df.drop_duplicates(subset=["mensaje"], keep="first").reset_index(drop=True)
print("Filas sin duplicados:", len(datos))
display(datos["etiqueta"].value_counts().to_frame("cantidad"))

RESUMEN["filas_leidas"] = int(len(df))
RESUMEN["filas_unicas"] = int(len(datos))
RESUMEN["duplicados_eliminados"] = int(len(df) - len(datos))
RESUMEN["ham_total"] = int((datos.etiqueta == "ham").sum())
RESUMEN["spam_total"] = int((datos.etiqueta == "spam").sum())

## 3. Partición estratificada (idéntica al Corte 2)
70 % entrenamiento, 15 % validación, 15 % prueba. Se verifica que los tamaños coincidan con el Corte 2 (3.618 / 775 / 776).

In [ ]:
X = datos["mensaje"].astype(str)
y = datos["etiqueta"].astype(str)

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED)

particion = pd.DataFrame({
    "particion": ["entrenamiento", "validación", "prueba (reservada)"],
    "n":    [len(y_train), len(y_val), len(y_test)],
    "ham":  [(y_train == "ham").sum(), (y_val == "ham").sum(), (y_test == "ham").sum()],
    "spam": [(y_train == "spam").sum(), (y_val == "spam").sum(), (y_test == "spam").sum()],
})
display(particion)
particion.to_csv(REPORTS / "particiones.csv", index=False)
if (len(y_train), len(y_val), len(y_test)) != (3618, 775, 776):
    print("ATENCIÓN: los tamaños no coinciden con el Corte 2. Revisa la lectura del archivo.")

# Variables binarias (spam = 1) para curvas y probabilidades
yb_train = (y_train == "spam").astype(int).values
yb_val   = (y_val   == "spam").astype(int).values
yb_test  = (y_test  == "spam").astype(int).values

# Train + validación: se usa SOLO después de fijar todas las decisiones
X_trval  = pd.concat([X_train, X_val])
y_trval  = pd.concat([y_train, y_val])
yb_trval = (y_trval == "spam").astype(int).values
RESUMEN["particion"] = particion.to_dict(orient="records")

## 4. Modelos de referencia que se van a comparar al final
- **Línea base del Corte 1:** clase mayoritaria y regla por palabras clave.
- **Modelo simple interpretable:** TF-IDF (1,1) + Naive Bayes.
- **Modelo provisional del Corte 2:** TF-IDF (1,2), `min_df=2`, `sublinear_tf=True` + SVM lineal, entrenado solo con entrenamiento.

In [ ]:
def metricas(y_true, y_pred, puntaje=None):
    cm = confusion_matrix(y_true, y_pred, labels=["ham", "spam"])
    out = {
        "precision_spam": precision_score(y_true, y_pred, pos_label="spam", zero_division=0),
        "recall_spam":    recall_score(y_true, y_pred, pos_label="spam", zero_division=0),
        "f1_spam":        f1_score(y_true, y_pred, pos_label="spam", zero_division=0),
        "accuracy":       accuracy_score(y_true, y_pred),
        "FP_ham_bloqueado": int(cm[0, 1]),
        "FN_spam_que_pasa": int(cm[1, 0]),
    }
    if puntaje is not None:
        out["pr_auc"] = average_precision_score((np.asarray(y_true) == "spam").astype(int), puntaje)
    return out

class ReglaPalabrasClave(BaseEstimator, ClassifierMixin):
    # Misma regla de la línea base del Corte 1
    def __init__(self, keywords=("free", "win", "winner", "prize", "claim", "urgent",
                                 "cash", "award", "click", "txt", "unsubscribe")):
        self.keywords = keywords
    def fit(self, X, y=None):
        self.classes_ = np.array(["ham", "spam"]); return self
    def predict(self, X):
        return np.array(["spam" if any(k in str(t).lower() for k in self.keywords) else "ham" for t in X])

def tfidf_palabras(ngram=(1, 2), min_df=2):
    return TfidfVectorizer(lowercase=True, ngram_range=ngram, min_df=min_df, sublinear_tf=True)

dummy = DummyClassifier(strategy="most_frequent").fit(np.zeros((len(y_train), 1)), y_train)
regla = ReglaPalabrasClave().fit(X_train)
nb_simple = Pipeline([("tfidf", tfidf_palabras((1, 1), 1)), ("clf", MultinomialNB())]).fit(X_train, y_train)
svm_corte2 = Pipeline([("tfidf", tfidf_palabras((1, 2), 2)),
                       ("clasificador", LinearSVC(random_state=SEED))]).fit(X_train, y_train)

ref_val = pd.DataFrame([
    {"modelo": "Dummy mayoritario (Corte 1)", **metricas(y_val, dummy.predict(np.zeros((len(y_val), 1))))},
    {"modelo": "Regla palabras clave (Corte 1)", **metricas(y_val, regla.predict(X_val))},
    {"modelo": "TF-IDF + Naive Bayes", **metricas(y_val, nb_simple.predict(X_val), nb_simple.predict_proba(X_val)[:, 1])},
    {"modelo": "SVM lineal provisional (Corte 2)", **metricas(y_val, svm_corte2.predict(X_val), svm_corte2.decision_function(X_val))},
])
display(ref_val.round(4))
print("Control: el SVM del Corte 2 debe dar F1 spam ≈ 0.9247 en validación.")

## 5. Profundización A — ¿Mejora la representación del texto?
El spam suele **ofuscar palabras** ("FR33", "W1N", "c-a-s-h"). Los n-gramas de palabras no ven eso; los **n-gramas de caracteres** sí. Se comparan, con el mismo clasificador (SVM lineal) y en validación:

1. Palabras (1,2) — la representación del Corte 2.
2. Caracteres `char_wb` (2,5).
3. Palabras (1,2) + caracteres (2,5) combinados.

Además, una **comparación acotada con un modelo moderno**: embeddings de oraciones (`all-MiniLM-L6-v2`) + regresión logística. Se usa como referencia, no como candidato final, porque el proyecto debe entrenarse y evaluarse localmente con un modelo liviano y explicable.

**Regla de selección (fijada antes de mirar resultados):** gana la representación TF-IDF con mayor **PR-AUC** en validación; si la ventaja sobre la del Corte 2 es menor a 0,005, se conserva la del Corte 2 porque es más simple.

In [ ]:
def tfidf_caracteres():
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True, lowercase=True)

REPRESENTACIONES = {
    "Palabras (1,2) [Corte 2]": lambda: tfidf_palabras((1, 2), 2),
    "Caracteres (2,5)":         lambda: tfidf_caracteres(),
    "Palabras + caracteres":    lambda: FeatureUnion([("pal", tfidf_palabras((1, 2), 2)),
                                                       ("car", tfidf_caracteres())]),
}

filas, pipes_rep = [], {}
for nombre, fabrica in REPRESENTACIONES.items():
    p = Pipeline([("vect", fabrica()), ("clf", LinearSVC(random_state=SEED))]).fit(X_train, y_train)
    pipes_rep[nombre] = p
    filas.append({"representacion": nombre, "n_caracteristicas": len(p.named_steps["vect"].get_feature_names_out()),
                  **metricas(y_val, p.predict(X_val), p.decision_function(X_val))})

# --- Comparación acotada con un modelo moderno (opcional) ---
EMB_OK = False
try:
    from sentence_transformers import SentenceTransformer
    st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cpu")
    E_train = st.encode(X_train.tolist(), batch_size=64, show_progress_bar=False, normalize_embeddings=True)
    E_val   = st.encode(X_val.tolist(),   batch_size=64, show_progress_bar=False, normalize_embeddings=True)
    lr_emb = LogisticRegression(max_iter=3000, C=10, random_state=SEED).fit(E_train, y_train)
    filas.append({"representacion": "Embeddings MiniLM + Reg. logística (referencia)",
                  "n_caracteristicas": E_train.shape[1],
                  **metricas(y_val, lr_emb.predict(E_val), lr_emb.predict_proba(E_val)[:, 1])})
    EMB_OK = True
except Exception as e:
    print("Comparación con embeddings omitida:", type(e).__name__, str(e)[:120])

comp_rep = pd.DataFrame(filas)
display(comp_rep.round(4))
comp_rep.to_csv(REPORTS / "profundizacion_representaciones_validacion.csv", index=False)

# --- Selección con la regla fijada ---
candidatas = comp_rep[comp_rep["representacion"].isin(REPRESENTACIONES)].set_index("representacion")
base_auc = candidatas.loc["Palabras (1,2) [Corte 2]", "pr_auc"]
mejor = candidatas["pr_auc"].idxmax()
REP_FINAL = mejor if candidatas.loc[mejor, "pr_auc"] - base_auc >= 0.005 else "Palabras (1,2) [Corte 2]"
print("Representación elegida para el modelo final:", REP_FINAL)
RESUMEN["representaciones_validacion"] = comp_rep.round(4).to_dict(orient="records")
RESUMEN["representacion_final"] = REP_FINAL
RESUMEN["embeddings_evaluados"] = EMB_OK

fig, ax = plt.subplots(figsize=(8, 3.2))
orden = comp_rep.sort_values("pr_auc")
ax.barh(orden["representacion"], orden["pr_auc"],
        color=[C_SPAM if r == REP_FINAL else C_GRIS for r in orden["representacion"]])
for i, v in enumerate(orden["pr_auc"]):
    ax.text(v + 0.002, i, f"{v:.3f}", va="center")
ax.set_xlim(max(0, orden["pr_auc"].min() - 0.05), 1.0)
ax.set_xlabel("PR-AUC spam (validación)")
ax.set_title("Representaciones comparadas en validación")
guardar_fig("05_representaciones.png")

## 6. Profundización B — Calibración de probabilidades
El SVM lineal entrega un puntaje, no una probabilidad. Para poder decir *"este mensaje tiene 93 % de probabilidad de ser spam"* y para que el umbral tenga sentido, se calibra con **Platt scaling** (`CalibratedClassifierCV`, método sigmoide, 5 folds internos).

Se mide con:
- **Brier score** (menor es mejor): error cuadrático medio de la probabilidad.
- **Curva de confiabilidad:** si el modelo dice 80 %, ¿realmente ~80 % de esos mensajes son spam?

In [ ]:
def pipeline_final_sin_calibrar():
    return Pipeline([("vect", REPRESENTACIONES[REP_FINAL]()), ("clf", LinearSVC(random_state=SEED))])

def pipeline_calibrado():
    return CalibratedClassifierCV(pipeline_final_sin_calibrar(), method="sigmoid", cv=5)

cal_train = pipeline_calibrado().fit(X_train, y_train)
p_val = cal_train.predict_proba(X_val)[:, list(cal_train.classes_).index("spam")]

# Puntaje del SVM sin calibrar llevado a [0,1] con min-max, solo para comparar (no es una probabilidad)
svm_sc = pipes_rep[REP_FINAL].decision_function(X_val)
svm_minmax = (svm_sc - svm_sc.min()) / (svm_sc.max() - svm_sc.min())

calib = pd.DataFrame([
    {"salida": "SVM sin calibrar (puntaje reescalado)", "brier": brier_score_loss(yb_val, svm_minmax)},
    {"salida": "SVM calibrado (Platt)",                 "brier": brier_score_loss(yb_val, p_val)},
])
display(calib.round(4))
RESUMEN["calibracion_validacion"] = calib.round(4).to_dict(orient="records")

fig, ax = plt.subplots(figsize=(4.6, 4.2))
fr, mp = calibration_curve(yb_val, p_val, n_bins=8, strategy="quantile")
ax.plot([0, 1], [0, 1], "--", color=C_GRIS, label="Calibración perfecta")
ax.plot(mp, fr, "o-", color=C_SPAM, label="SVM calibrado")
ax.set_xlabel("Probabilidad predicha de spam"); ax.set_ylabel("Fracción real de spam")
ax.set_title("Curva de confiabilidad (validación)"); ax.legend(loc="upper left")
guardar_fig("06_calibracion_validacion.png")

## 7. Profundización C — Umbral de decisión según el costo del error
**¿Qué error es más costoso?** Bloquear un mensaje legítimo (falso positivo) puede hacer perder un código de verificación, una cita médica o un mensaje familiar. Dejar pasar un spam es molesto pero el usuario lo puede ignorar. Por eso se asume que **un falso positivo cuesta 5 veces más que un falso negativo** (`COSTO_FP = 5`, `COSTO_FN = 1`). Es un supuesto del proyecto y se analiza su sensibilidad.

El umbral se elige **sin tocar el test**, sobre 4.393 mensajes:
- predicciones *out-of-fold* (validación cruzada estratificada de 5 folds) en entrenamiento, más
- las predicciones de validación del modelo entrenado con entrenamiento.

Se elige el umbral que minimiza el costo total; si hay empate, el más alto (más conservador con el ham).

In [ ]:
COSTO_FP, COSTO_FN = 5, 1
UMBRALES = np.round(np.arange(0.05, 0.96, 0.01), 2)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
p_oof = cross_val_predict(pipeline_calibrado(), X_train, y_train, cv=skf, method="predict_proba")[:, 1]
# (orden de clases alfabético: ham=0, spam=1)

p_sel  = np.concatenate([p_oof, p_val])
yb_sel = np.concatenate([yb_train, yb_val])

def barrido(yb, p, umbrales=UMBRALES, cfp=COSTO_FP, cfn=COSTO_FN):
    filas = []
    for t in umbrales:
        pred = (p >= t).astype(int)
        fp = int(((pred == 1) & (yb == 0)).sum()); fn = int(((pred == 0) & (yb == 1)).sum())
        tp = int(((pred == 1) & (yb == 1)).sum())
        prec = tp / (tp + fp) if tp + fp else 0.0; rec = tp / (tp + fn) if tp + fn else 0.0
        filas.append({"umbral": t, "FP_ham_bloqueado": fp, "FN_spam_que_pasa": fn,
                      "precision_spam": prec, "recall_spam": rec,
                      "f1_spam": 2 * prec * rec / (prec + rec) if prec + rec else 0.0,
                      "costo": cfp * fp + cfn * fn})
    return pd.DataFrame(filas)

tabla_umbral = barrido(yb_sel, p_sel)
costo_min = tabla_umbral["costo"].min()
UMBRAL = float(tabla_umbral[tabla_umbral["costo"] == costo_min]["umbral"].max())
tabla_umbral.to_csv(REPORTS / "barrido_umbral_seleccion.csv", index=False)
print(f"Umbral elegido (costo FP={COSTO_FP}, FN={COSTO_FN}): {UMBRAL:.2f}")
display(tabla_umbral[tabla_umbral["umbral"].isin([0.3, 0.5, UMBRAL, 0.7, 0.9])].round(4))

# Sensibilidad: ¿cómo cambia el umbral si el costo relativo del FP fuera otro?
sens = []
for cfp in [1, 2, 5, 10, 20]:
    tb = barrido(yb_sel, p_sel, cfp=cfp)
    t = float(tb[tb["costo"] == tb["costo"].min()]["umbral"].max())
    f = tb[tb["umbral"] == t].iloc[0]
    sens.append({"costo_FP_vs_FN": f"{cfp}:1", "umbral": t, "FP": int(f.FP_ham_bloqueado),
                 "FN": int(f.FN_spam_que_pasa), "precision_spam": f.precision_spam, "recall_spam": f.recall_spam})
sens = pd.DataFrame(sens)
display(sens.round(4))
sens.to_csv(REPORTS / "sensibilidad_costo_umbral.csv", index=False)
RESUMEN["umbral"] = UMBRAL
RESUMEN["costos"] = {"FP": COSTO_FP, "FN": COSTO_FN}
RESUMEN["sensibilidad_costo"] = sens.round(4).to_dict(orient="records")

fig, ax1 = plt.subplots(figsize=(8, 3.6))
ax1.plot(tabla_umbral.umbral, tabla_umbral.precision_spam, color=C_HAM, lw=2, label="Precision spam")
ax1.plot(tabla_umbral.umbral, tabla_umbral.recall_spam, color=C_SPAM, lw=2, label="Recall spam")
ax1.axvline(0.5, color=C_GRIS, ls=":", label="Umbral por defecto 0,5")
ax1.axvline(UMBRAL, color=C_OSC, ls="--", label=f"Umbral elegido {UMBRAL:.2f}")
ax1.set_xlabel("Umbral sobre la probabilidad de spam"); ax1.set_ylabel("Métrica")
ax1.set_ylim(0.5, 1.01); ax1.legend(loc="lower left", fontsize=8)
ax2 = ax1.twinx(); ax2.plot(tabla_umbral.umbral, tabla_umbral.costo, color=C_GRIS, alpha=.6)
ax2.set_ylabel("Costo (5·FP + 1·FN)", color=C_GRIS); ax2.spines["right"].set_visible(True)
ax1.set_title("Selección del umbral (train out-of-fold + validación)")
guardar_fig("07_seleccion_umbral.png")

## 8. Evaluación final en el test reservado
Con todo fijado (representación, calibración y umbral), el modelo final se reentrena con **entrenamiento + validación** y se evalúa **una sola vez** en el test (776 mensajes).

Se compara contra:
- la línea base del Corte 1 (mayoritaria y regla),
- el modelo simple TF-IDF + Naive Bayes,
- el modelo provisional del Corte 2 (tal como quedó, entrenado con entrenamiento),
- el modelo final con umbral 0,5, para aislar el efecto del umbral.

In [ ]:
modelo_final = pipeline_calibrado().fit(X_trval, y_trval)
idx_spam = list(modelo_final.classes_).index("spam")
p_test = modelo_final.predict_proba(X_test)[:, idx_spam]
pred_final = np.where(p_test >= UMBRAL, "spam", "ham")
pred_final_05 = np.where(p_test >= 0.5, "spam", "ham")

comparacion_test = pd.DataFrame([
    {"modelo": "Dummy mayoritario (Corte 1)", **metricas(y_test, dummy.predict(np.zeros((len(y_test), 1))))},
    {"modelo": "Regla palabras clave (Corte 1)", **metricas(y_test, regla.predict(X_test))},
    {"modelo": "TF-IDF + Naive Bayes", **metricas(y_test, nb_simple.predict(X_test), nb_simple.predict_proba(X_test)[:, 1])},
    {"modelo": "SVM provisional (Corte 2)", **metricas(y_test, svm_corte2.predict(X_test), svm_corte2.decision_function(X_test))},
    {"modelo": "Final calibrado, umbral 0,5", **metricas(y_test, pred_final_05, p_test)},
    {"modelo": f"FINAL calibrado, umbral {UMBRAL:.2f}", **metricas(y_test, pred_final, p_test)},
])
display(comparacion_test.round(4))
comparacion_test.to_csv(REPORTS / "comparacion_final_test.csv", index=False)
RESUMEN["comparacion_test"] = comparacion_test.round(4).to_dict(orient="records")
RESUMEN["brier_test_final"] = round(float(brier_score_loss(yb_test, p_test)), 4)
print(classification_report(y_test, pred_final, labels=["ham", "spam"], digits=4, zero_division=0))

# Matriz de confusión del modelo final
cm = confusion_matrix(y_test, pred_final, labels=["ham", "spam"])
RESUMEN["matriz_confusion_test_final"] = {"ham_ok": int(cm[0, 0]), "FP": int(cm[0, 1]), "FN": int(cm[1, 0]), "spam_ok": int(cm[1, 1])}
fig, ax = plt.subplots(figsize=(4.4, 3.8))
colores = np.array([[C_HAM, C_SPAM], [C_SPAM, C_OSC]])
etiquetas = np.array([["Ham correcto", "Ham bloqueado (FP)"], ["Spam que pasa (FN)", "Spam detectado"]])
for i in range(2):
    for j in range(2):
        ax.add_patch(plt.Rectangle((j, 1 - i), 1, 1, color=colores[i, j], alpha=.9 if i == j else .25))
        ax.text(j + .5, 1 - i + .58, str(cm[i, j]), ha="center", va="center", fontsize=22, fontweight="bold",
                color="white" if i == j else C_OSC)
        ax.text(j + .5, 1 - i + .28, etiquetas[i, j], ha="center", va="center", fontsize=8.5,
                color="white" if i == j else C_OSC)
ax.set_xlim(0, 2); ax.set_ylim(0, 2); ax.set_xticks([.5, 1.5]); ax.set_xticklabels(["Predicho ham", "Predicho spam"])
ax.set_yticks([1.5, .5]); ax.set_yticklabels(["Real ham", "Real spam"])
ax.set_title(f"Modelo final en test (umbral {UMBRAL:.2f})")
guardar_fig("08_matriz_confusion_final_test.png")

# Comparación visual
fig, ax = plt.subplots(figsize=(9, 3.6))
cmp = comparacion_test.iloc[1:].copy()
xx = np.arange(len(cmp)); w = .27
ax.bar(xx - w, cmp.precision_spam, w, color=C_HAM, label="Precision spam")
ax.bar(xx, cmp.recall_spam, w, color=C_SPAM, label="Recall spam")
ax.bar(xx + w, cmp.f1_spam, w, color=C_OSC, label="F1 spam")
ax.set_xticks(xx); ax.set_xticklabels([textwrap.fill(m, 18) for m in cmp.modelo], fontsize=8)
ax.set_ylim(0, 1.05); ax.legend(ncol=3, loc="lower right", fontsize=8)
ax.set_title("Test reservado: línea base vs. Corte 2 vs. modelo final")
guardar_fig("08_comparacion_test.png")

# Curva precisión-recall en test
fig, ax = plt.subplots(figsize=(4.6, 4))
for nombre, s, c in [("Final calibrado", p_test, C_SPAM), ("SVM Corte 2", svm_corte2.decision_function(X_test), C_GRIS)]:
    pr, rc, _ = precision_recall_curve(yb_test, s)
    ax.plot(rc, pr, color=c, lw=2, label=f"{nombre} (AP={average_precision_score(yb_test, s):.3f})")
ax.set_xlabel("Recall spam"); ax.set_ylabel("Precision spam"); ax.set_ylim(0.5, 1.01)
ax.set_title("Curva precisión-recall (test)"); ax.legend(loc="lower left", fontsize=8)
guardar_fig("08_curva_pr_test.png")

### 8.1 Errores reales del test (lo que la defensa debe mostrar)

In [ ]:
errores_test = pd.DataFrame({"mensaje": X_test.values, "real": y_test.values,
                             "predicho": pred_final, "prob_spam": p_test.round(4)})
errores_test = errores_test[errores_test.real != errores_test.predicho].copy()
errores_test["tipo_error"] = np.where(errores_test.real == "ham", "Falso positivo (ham bloqueado)",
                                      "Falso negativo (spam que pasa)")
errores_test = errores_test.sort_values(["tipo_error", "prob_spam"], ascending=[False, False])
pd.set_option("display.max_colwidth", 160)
display(errores_test[["tipo_error", "prob_spam", "mensaje"]])
errores_test.to_csv(REPORTS / "errores_test_final.csv", index=False, encoding="utf-8-sig")

# Aciertos con probabilidad muy alta / muy baja, útiles como ejemplos en la presentación
ejemplos = pd.DataFrame({"mensaje": X_test.values, "real": y_test.values, "prob_spam": p_test.round(4)})
ej_ok = pd.concat([ejemplos[ejemplos.real == "spam"].nlargest(3, "prob_spam"),
                   ejemplos[ejemplos.real == "ham"].nsmallest(3, "prob_spam")])
display(ej_ok)
ej_ok.to_csv(REPORTS / "ejemplos_aciertos_test.csv", index=False, encoding="utf-8-sig")
RESUMEN["errores_test"] = errores_test[["tipo_error", "prob_spam", "mensaje"]].to_dict(orient="records")
RESUMEN["ejemplos_aciertos_test"] = ej_ok.to_dict(orient="records")

## 9. ¿Qué cambia al ajustar el umbral? (ilustrativo en test)
La guía pide mostrarlo en la presentación final. **El umbral ya estaba fijado en la sección 7**; esta tabla solo ilustra el compromiso entre bloquear ham y dejar pasar spam. No se usa para decidir nada.

In [ ]:
umbral_test = barrido(yb_test, p_test, umbrales=np.array(sorted({0.2, 0.3, 0.4, 0.5, UMBRAL, 0.6, 0.7, 0.8, 0.9})))
display(umbral_test.round(4))
umbral_test.to_csv(REPORTS / "efecto_umbral_test_ilustrativo.csv", index=False)
RESUMEN["efecto_umbral_test"] = umbral_test.round(4).to_dict(orient="records")

fig, ax = plt.subplots(figsize=(7.5, 3.4))
ut = barrido(yb_test, p_test, umbrales=np.round(np.arange(0.05, 0.96, 0.01), 2))
ax.plot(ut.umbral, ut.FP_ham_bloqueado, color=C_HAM, lw=2.2, label="Ham bloqueado (FP)")
ax.plot(ut.umbral, ut.FN_spam_que_pasa, color=C_SPAM, lw=2.2, label="Spam que pasa (FN)")
ax.axvline(UMBRAL, color=C_OSC, ls="--", label=f"Umbral elegido {UMBRAL:.2f}")
ax.set_xlabel("Umbral"); ax.set_ylabel("Cantidad de mensajes (test)")
ax.set_title("Mover el umbral cambia qué error se comete"); ax.legend(fontsize=8)
guardar_fig("09_efecto_umbral_test.png")

## 10. Términos que dominan las decisiones y posible sesgo temporal
Se entrena un SVM sin calibrar con la misma representación final sobre entrenamiento + validación para leer sus coeficientes (positivo → empuja a spam). Se marcan términos ligados a la **época del corpus** (2000–2004, Reino Unido): marcas de teléfonos y servicios de esa época, tonos de llamada, números premium, libras esterlinas. Si el modelo depende de ellos, puede fallar con el spam actual.

In [ ]:
interp = pipeline_final_sin_calibrar().fit(X_trval, y_trval)
nombres = interp.named_steps["vect"].get_feature_names_out()
coef = interp.named_steps["clf"].coef_.ravel()
terminos = pd.DataFrame({"termino": nombres, "coeficiente": coef})
terminos["termino"] = terminos["termino"].str.replace(r"^(pal|car)__", "", regex=True)

PATRON_EPOCA = re.compile(r"(ringtone|tone|polyphonic|nokia|motorola|orange|vodafone|wap|mms|£|\bp\b|150p|"
                          r"\bmob\b|\b0[89]\d{2,}|\b8\d{4}\b|200[0-9]|\bpobox)", re.I)
terminos["posible_sesgo_temporal"] = terminos["termino"].apply(lambda t: bool(PATRON_EPOCA.search(t)))
top_spam = terminos.nlargest(20, "coeficiente").reset_index(drop=True)
top_ham  = terminos.nsmallest(20, "coeficiente").reset_index(drop=True)
display(top_spam.round(4)); display(top_ham.round(4))
top_spam.to_csv(REPORTS / "terminos_spam_final.csv", index=False)
top_ham.to_csv(REPORTS / "terminos_ham_final.csv", index=False)
RESUMEN["top_terminos_spam"] = top_spam.round(4).to_dict(orient="records")
RESUMEN["top_terminos_ham"] = top_ham.round(4).to_dict(orient="records")
RESUMEN["pct_top20_spam_con_sesgo_temporal"] = float(top_spam.posible_sesgo_temporal.mean())

fig, ax = plt.subplots(figsize=(8, 5))
t15 = pd.concat([top_ham.head(12), top_spam.head(12)]).sort_values("coeficiente")
cols = [C_SPAM if c > 0 else C_HAM for c in t15.coeficiente]
bars = ax.barh([repr(t)[1:-1] for t in t15.termino], t15.coeficiente, color=cols)
for b, s in zip(bars, t15.posible_sesgo_temporal):
    if s: b.set_hatch("///"); b.set_edgecolor("white")
ax.axvline(0, color=C_OSC, lw=.8)
ax.set_xlabel("Coeficiente (← ham | spam →)")
ax.set_title("Términos con más peso (rayado = posible sesgo de época)")
guardar_fig("10_terminos.png")

## 11. Pruebas de robustez y sensibilidad
**11.1 Perturbaciones sobre el test.** Se modifican los mensajes del test como lo haría un spammer o un usuario y se mide cuánto cambia el desempeño (mismo modelo, mismo umbral):
- todo en MAYÚSCULAS,
- ofuscación tipo *leetspeak* (o→0, e→3, i→1, a→4),
- palabras partidas con puntos ("f.r.e.e"),
- sin números (se quitan teléfonos y códigos cortos).

**11.2 Fuera de dominio.** Mensajes escritos por el equipo, en español y con estrategias de spam actuales (paquetería, bancos, cripto). Es una demostración, no una evaluación estadística: el corpus es en inglés y de 2000–2004.

In [ ]:
rng = np.random.default_rng(SEED)
LEET = str.maketrans({"o": "0", "O": "0", "e": "3", "E": "3", "i": "1", "I": "1", "a": "4", "A": "4"})

def leet(t):      return " ".join(w.translate(LEET) if rng.random() < .5 else w for w in t.split())
def puntos(t):    return " ".join(".".join(w) if (len(w) > 3 and rng.random() < .3) else w for w in t.split())
def sin_num(t):   return re.sub(r"\d+", "", t)

PERTURBACIONES = {"Original": lambda t: t, "MAYÚSCULAS": str.upper, "Leetspeak": leet,
                  "Palabras con puntos": puntos, "Sin números": sin_num}

# Cada perturbación se aplica UNA vez, así ambos modelos ven exactamente los mismos textos
X_PERTURBADO = {nom: X_test.map(f) for nom, f in PERTURBACIONES.items()}

def evaluar_perturbado(modelo_prob, umbral, nombre_modelo):
    filas = []
    for nom, Xp in X_PERTURBADO.items():
        pp = modelo_prob(Xp)
        pred = np.where(pp >= umbral, "spam", "ham")
        filas.append({"modelo": nombre_modelo, "perturbacion": nom, **metricas(y_test, pred, pp)})
    return filas

rob = evaluar_perturbado(lambda Z: modelo_final.predict_proba(Z)[:, idx_spam], UMBRAL, "Final")
rob += evaluar_perturbado(lambda Z: svm_corte2.decision_function(Z), 0.0, "SVM Corte 2")
robustez = pd.DataFrame(rob)
display(robustez[["modelo", "perturbacion", "precision_spam", "recall_spam", "f1_spam",
                  "FP_ham_bloqueado", "FN_spam_que_pasa"]].round(4))
robustez.to_csv(REPORTS / "robustez_perturbaciones_test.csv", index=False)
RESUMEN["robustez"] = robustez.round(4).to_dict(orient="records")

fig, ax = plt.subplots(figsize=(8, 3.4))
piv = robustez.pivot(index="perturbacion", columns="modelo", values="recall_spam").loc[list(PERTURBACIONES)]
xx = np.arange(len(piv)); w = .38
ax.bar(xx - w/2, piv["SVM Corte 2"], w, color=C_GRIS, label="SVM Corte 2")
ax.bar(xx + w/2, piv["Final"], w, color=C_SPAM, label="Modelo final")
for i, (a, b) in enumerate(zip(piv["SVM Corte 2"], piv["Final"])):
    ax.text(i - w/2, a + .01, f"{a:.2f}", ha="center", fontsize=8); ax.text(i + w/2, b + .01, f"{b:.2f}", ha="center", fontsize=8)
ax.set_xticks(xx); ax.set_xticklabels(piv.index); ax.set_ylim(0, 1.18)
ax.set_ylabel("Recall spam (test perturbado)"); ax.legend(fontsize=8, ncol=2, loc="upper center")
ax.set_title("Robustez ante ofuscación")
guardar_fig("11_robustez.png")

In [ ]:
FUERA_DE_DOMINIO = [
    # (mensaje, etiqueta esperada por el equipo, grupo)
    ("Hola ma, ya llegué a la casa. Te llamo en un rato", "ham", "Español"),
    ("Profe, mañana no puedo ir a clase, le envío el trabajo por correo", "ham", "Español"),
    ("Tu código de verificación es 482913. No lo compartas con nadie", "ham", "Español"),
    ("¡Felicidades! Ganaste un iPhone. Reclámalo ya en bit.ly/premio-col", "spam", "Español"),
    ("BANCO: su cuenta fue bloqueada. Ingrese a www.banco-seguro-co.com para validar sus datos", "spam", "Español"),
    ("Su paquete no pudo ser entregado. Pague $2.500 de envío aquí: entregas-co.info", "spam", "Español"),
    ("Your package is on hold due to unpaid customs fee. Pay now: dhl-track-help.com", "spam", "Spam actual (inglés)"),
    ("Your Netflix account is suspended. Update your payment details at netflx-billing.com", "spam", "Spam actual (inglés)"),
    ("Earn $500 a day trading crypto with our AI bot. Join our WhatsApp group now", "spam", "Spam actual (inglés)"),
    ("Hey, are we still meeting for lunch today?", "ham", "Inglés cotidiano"),
    ("Can you send me the slides before the meeting tomorrow?", "ham", "Inglés cotidiano"),
    ("Congratulations! You have won a free prize. Click now to claim.", "spam", "Inglés estilo corpus"),
]
fd = pd.DataFrame(FUERA_DE_DOMINIO, columns=["mensaje", "esperado", "grupo"])
fd["prob_spam"] = modelo_final.predict_proba(fd.mensaje)[:, idx_spam].round(4)
fd["predicho"] = np.where(fd.prob_spam >= UMBRAL, "spam", "ham")
fd["acierto"] = fd.predicho == fd.esperado
display(fd)
display(fd.groupby("grupo")["acierto"].agg(["sum", "count"]))
fd.to_csv(REPORTS / "prueba_fuera_de_dominio.csv", index=False, encoding="utf-8-sig")
RESUMEN["fuera_de_dominio"] = fd.to_dict(orient="records")

## 12. Función de inferencia para mensajes nuevos + demo
Se guarda el modelo final (vectorizador + SVM calibrado) y el umbral en `models/`, y se escribe `src/inferencia.py` para usarlo desde cualquier script:

```python
from src.inferencia import clasificar_sms
clasificar_sms("WINNER!! Claim your prize now")
```

In [ ]:
paquete = {"modelo": modelo_final, "umbral": UMBRAL, "representacion": REP_FINAL,
           "costos": {"FP": COSTO_FP, "FN": COSTO_FN}, "sklearn": sklearn.__version__,
           "entrenado_con": "train + validación (4.393 mensajes), UCI SMS Spam Collection"}
joblib.dump(paquete, MODELS / "modelo_final_sms.joblib")

(SRC / "__init__.py").write_text("")
(SRC / "inferencia.py").write_text('''# Inferencia del clasificador de SMS (Proyecto 8, Corte 3)
from pathlib import Path
import joblib

_RUTA = Path(__file__).resolve().parent.parent / "models" / "modelo_final_sms.joblib"
_PAQ = None

def _cargar():
    global _PAQ
    if _PAQ is None:
        _PAQ = joblib.load(_RUTA)
    return _PAQ

def clasificar_sms(texto, umbral=None):
    # Devuelve etiqueta (ham/spam), probabilidad de spam y el umbral usado
    paq = _cargar()
    if not isinstance(texto, str) or not texto.strip():
        raise ValueError("El mensaje debe ser un texto no vacío.")
    t = paq["umbral"] if umbral is None else float(umbral)
    modelo = paq["modelo"]
    p = float(modelo.predict_proba([texto])[0][list(modelo.classes_).index("spam")])
    return {"mensaje": texto, "etiqueta": "spam" if p >= t else "ham",
            "prob_spam": round(p, 4), "umbral": t,
            "aviso": "Modelo entrenado con SMS en inglés (2000-2004). No validado en otros idiomas."}

if __name__ == "__main__":
    import sys
    for m in (sys.argv[1:] or ["Hey, are we still meeting for lunch today?"]):
        print(clasificar_sms(m))
''', encoding="utf-8")

import importlib, sys
sys.path.insert(0, str(BASE))
import src.inferencia as inf; importlib.reload(inf)
for m in ["Hey, are we still meeting for lunch today?",
          "URGENT! Your mobile number has won £2000. Call 09061701461 to claim",
          "Ok lar, see you at 7 then"]:
    print(inf.clasificar_sms(m))

In [ ]:
# DEMO en vivo para la sustentación: escribe un mensaje y presiona Enter (deja vacío para salir)
try:
    import ipywidgets as w
    caja = w.Text(placeholder="Escribe un SMS…", layout=w.Layout(width="70%"))
    deslizador = w.FloatSlider(value=UMBRAL, min=0.05, max=0.95, step=0.05, description="Umbral")
    salida = w.Output()
    def _clasificar(_=None):
        salida.clear_output()
        if caja.value.strip():
            r = inf.clasificar_sms(caja.value, umbral=deslizador.value)
            color = C_SPAM if r["etiqueta"] == "spam" else C_HAM
            with salida:
                from IPython.display import HTML, display as _d
                _d(HTML(f"<div style='font-size:18px'><b style='color:{color}'>{r['etiqueta'].upper()}</b>"
                        f" · probabilidad de spam {r['prob_spam']:.1%} · umbral {r['umbral']:.2f}</div>"))
    caja.on_submit(_clasificar); deslizador.observe(_clasificar, names="value")
    display(w.VBox([caja, deslizador, salida]))
except Exception as e:
    print("ipywidgets no disponible; usa inf.clasificar_sms('texto')", e)

## 13. Model card y ficha del dataset
Se generan automáticamente con los números de este notebook (`reports/MODEL_CARD.md` y `reports/DATASET_CARD.md`). **Revísenlas y ajusten la redacción** antes de entregar.

In [ ]:
fin = comparacion_test.iloc[-1]; c2 = comparacion_test[comparacion_test.modelo.str.contains("Corte 2")].iloc[0]
rg = comparacion_test[comparacion_test.modelo.str.contains("Regla")].iloc[0]
fd_es = fd[fd.grupo == "Español"]; fd_new = fd[fd.grupo == "Spam actual (inglés)"]
rob_f = robustez[robustez.modelo == "Final"].set_index("perturbacion")

model_card = f'''# Model card — Clasificador de SMS ham/spam (Proyecto 8, Corte 3)

## Descripción
- **Tarea:** clasificación binaria de SMS en *ham* (legítimo) o *spam* (no deseado).
- **Modelo:** TF-IDF ({REP_FINAL}) + SVM lineal calibrado con Platt scaling (`CalibratedClassifierCV`, sigmoide, 5 folds).
- **Decisión:** se marca spam si la probabilidad calibrada es ≥ **{UMBRAL:.2f}**. El umbral se eligió minimizando el costo 5·FP + 1·FN sobre predicciones out-of-fold de entrenamiento + validación, sin usar el test.
- **Entrenamiento final:** entrenamiento + validación (4.393 mensajes). scikit-learn {sklearn.__version__}.

## Uso previsto
- Apoyo académico y demostrativo para filtrar SMS en **inglés** con estilo similar al corpus.
- Recomendado como **filtro de sugerencia o cuarentena** (el usuario puede revisar), no como bloqueo automático definitivo.

## Usos no recomendados
- Bloquear mensajes automáticamente sin posibilidad de revisión.
- Mensajes en **español** u otros idiomas: no está validado ({int(fd_es.acierto.sum())}/{len(fd_es)} aciertos en la prueba ilustrativa en español).
- Spam moderno (smishing bancario, paquetería, cripto, enlaces acortados): {int(fd_new.acierto.sum())}/{len(fd_new)} aciertos en la prueba ilustrativa.
- Decisiones con consecuencias legales, financieras o de seguridad.

## Desempeño en test reservado (776 mensajes, 98 spam)
| Modelo | Precision spam | Recall spam | F1 spam | Ham bloqueado (FP) | Spam que pasa (FN) |
|---|---|---|---|---|---|
| Regla palabras clave (Corte 1) | {rg.precision_spam:.3f} | {rg.recall_spam:.3f} | {rg.f1_spam:.3f} | {rg.FP_ham_bloqueado} | {rg.FN_spam_que_pasa} |
| SVM provisional (Corte 2) | {c2.precision_spam:.3f} | {c2.recall_spam:.3f} | {c2.f1_spam:.3f} | {c2.FP_ham_bloqueado} | {c2.FN_spam_que_pasa} |
| **Final (umbral {UMBRAL:.2f})** | **{fin.precision_spam:.3f}** | **{fin.recall_spam:.3f}** | **{fin.f1_spam:.3f}** | **{fin.FP_ham_bloqueado}** | **{fin.FN_spam_que_pasa}** |

PR-AUC del modelo final en test: {fin.pr_auc:.3f}. Brier score: {RESUMEN["brier_test_final"]}.
Estos valores describen el desempeño **en este dataset**; en el mundo real se espera menor desempeño por cambio de idioma, época y estrategias de spam.

## Robustez (recall de spam en test perturbado)
Original {rob_f.loc["Original","recall_spam"]:.3f} · MAYÚSCULAS {rob_f.loc["MAYÚSCULAS","recall_spam"]:.3f} · Leetspeak {rob_f.loc["Leetspeak","recall_spam"]:.3f} · Palabras con puntos {rob_f.loc["Palabras con puntos","recall_spam"]:.3f} · Sin números {rob_f.loc["Sin números","recall_spam"]:.3f}

## Limitaciones y riesgos
- **Idioma:** corpus en inglés (Reino Unido y Singapur). No generaliza a español.
- **Época:** mensajes de aprox. 2000–2004. Varios términos de mayor peso son propios de esa época (tonos, números premium, libras); el spam actual usa enlaces, suplantación de marcas y bancos.
- **Cambio de estrategias de spam:** los spammers adaptan el texto; el modelo debe reentrenarse periódicamente con datos recientes y etiquetados.
- **Tamaño y desbalance:** solo 98 spam en test; una diferencia de 1–2 mensajes cambia la precisión o el recall en ~1–2 puntos.
- **Privacidad:** en un uso real los SMS son datos personales; no deben almacenarse ni enviarse a terceros sin consentimiento.
- **Calibración:** las probabilidades están calibradas para la proporción de spam del corpus (~12,6 %); si la proporción real cambia, el umbral debe revisarse.

## Condiciones de uso
Usar solo con SMS en inglés, como sugerencia o cuarentena revisable, citando el dataset y revisando periódicamente falsos positivos.
'''
(REPORTS / "MODEL_CARD.md").write_text(model_card, encoding="utf-8")

dataset_card = f'''# Ficha del dataset — SMS Spam Collection

- **Fuente canónica:** UCI Machine Learning Repository, "SMS Spam Collection", Almeida, T. & Hidalgo, J. (2011). DOI: 10.24432/C5CC84. Licencia CC BY 4.0.
- **Contenido:** SMS en inglés etiquetados como ham o spam. UCI documenta 5.574 mensajes; la lectura usada en el proyecto produce {RESUMEN["filas_leidas"]} filas.
- **Origen de los mensajes:** foro británico Grumbletext (spam), corpus NUS SMS de Singapur (ham), tesis doctoral de Caroline Tag y SMS Spam Corpus v.0.1.
- **Depuración:** se eliminaron {RESUMEN["duplicados_eliminados"]} textos duplicados exactos (conservando la primera aparición) para evitar fuga entre particiones. No hubo textos con etiquetas contradictorias. Quedan {RESUMEN["filas_unicas"]} mensajes: {RESUMEN["ham_total"]} ham y {RESUMEN["spam_total"]} spam ({RESUMEN["spam_total"]/RESUMEN["filas_unicas"]:.1%}).
- **Partición:** estratificada 70/15/15, `random_state=42`: 3.618 / 775 / 776.
- **Sesgos conocidos:** inglés británico y de Singapur; época 2000–2004; spam centrado en concursos, tonos de llamada y números premium; abreviaturas propias de SMS ("u", "ur", "txt").
- **Privacidad:** son mensajes reales, publicados para investigación; no se deben intentar re-identificar personas.
- **Uso en el proyecto:** entrenamiento, validación y test reservado de un clasificador local reproducible.
'''
(REPORTS / "DATASET_CARD.md").write_text(dataset_card, encoding="utf-8")
print(model_card[:1500], "...")

## 14. Paquete de resultados
Se guarda `reports/resumen_corte3.json` con todos los números y se comprime todo (figuras, tablas, model card, modelo e inferencia) en `resultados_corte3.zip`. En Colab se descarga automáticamente; **ese zip es el que se usa para armar la presentación**.

In [ ]:
(BASE / "requirements.txt").write_text(
    f"pandas\nnumpy\nscikit-learn=={sklearn.__version__}\nmatplotlib\njoblib\n"
    "# opcional para la comparación con embeddings\nsentence-transformers\n", encoding="utf-8")

with open(REPORTS / "resumen_corte3.json", "w", encoding="utf-8") as f:
    json.dump(RESUMEN, f, ensure_ascii=False, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))

zip_path = BASE / "resultados_corte3.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for carpeta in (REPORTS, MODELS, SRC):
        for f in carpeta.rglob("*"):
            if f.is_file() and "__pycache__" not in f.parts:
                z.write(f, f.relative_to(BASE))
    z.write(BASE / "requirements.txt", "requirements.txt")
print("Listo:", zip_path, f"({zip_path.stat().st_size/1e6:.1f} MB)")
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print("Descarga manual: panel de archivos → resultados_corte3.zip")